[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# DataFrame


## What you will be able to do

Build a table from a dictionary of columns, a list of rows, a list of records or a NumPy array, and
read its shape, columns, index and types before you trust what is in it.

## The idea

### The problem

Data reaches a program in more than one arrangement. A calculation produces columns: a list of station
names, a list of means, a list of latitudes. A loop or a file reader produces rows, one tuple at a
time, which is how Setup's `year_of_readings` hands over the year's 35,040 readings. An API sends
records, a dictionary for each station, as the **APIs and JSON** guide showed, and a record can leave
out a field the others have. NumPy produces an array with no names at all.

Each arrangement becomes a table differently, and pandas needs telling which parts are the names of
the columns and which are the rows. When the telling goes wrong, the table often still gets built. A
latitude typed as text in one row makes the whole column `object`, a column of Python objects. A record
with an extra field adds a column that is empty almost everywhere. The first sign of either is an answer
that comes out wrong several cells later.

So the habit worth forming is to look at a table's shape and types as soon as it exists, before asking
it anything.

### What a DataFrame is

> A **DataFrame** is a two-dimensional table of named columns, each a Series of one type, sharing one
> **index** of row labels. Its **shape** is the number of rows and the number of columns, and its
> **dtypes** give the type of each column.

### Why it works that way

A DataFrame is organized by column. Each column is one Series with one type, so a table can hold
station names as text, times as dates and readings as decimals side by side, and `dtypes` reports a
type for each of them. That is also why taking one column is quick and natural. A row cuts across the
columns, so when they hold different types pandas can only store it as general Python objects.

The constructor reads a dictionary as columns, its keys becoming the column names, so every list in it
must have the same length. It reads a list as rows, so it needs the column names given separately. A
list of dictionaries is rows whose keys name the columns, and a key that only some rows have becomes a
column with `NaN` in the rest. A NumPy array becomes rows and columns in the same shape, with one type
for the whole table, and takes its names from `columns` and `index`.

The type pandas calls **`object`** is the one to watch. It means the column holds Python objects one by
one, usually text. A column of numbers with a single text value among them becomes `object` too, and
arithmetic on it either fails or does something other than arithmetic.

### Where this shows up

Setup builds `readings` from a list of rows, and the **Why Pandas** notebook built a table from a
dictionary of two Series lined up by label. `read_csv`, in the **Reading Data** notebook, returns a
DataFrame, and so does a database query read with pandas. The NumPy arrays of the **NumPy, Deep Dive**
guide become tables here with names attached.

### What this notebook covers

- a table from a dictionary of columns, a list of rows, a list of records and a NumPy array
- `shape`, `columns`, `index` and `dtypes`, read before anything else
- a column as a Series that shares the table's index, and a row as a Series that has to hold every
  column's values
- the readings turned wide, one column for each station and one row for each hour
- a number stored as text, which builds without an error

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import pandas as pd

summary = pd.DataFrame({
    "station": ["Bergen", "Oslo"],
    "mean": [8.0, 6.5],
    "latitude": [60.39, 59.91],
})
print(summary)
print(summary.shape)
print(summary.dtypes)
```

```
  station  mean  latitude
0  Bergen   8.0     60.39
1    Oslo   6.5     59.91
(2, 3)
station      object
mean        float64
latitude    float64
dtype: object
```

Each key of the dictionary became a column, in order, and each list its values. The shape (2, 3) is two
rows and three columns, and `dtypes` shows text for the names, stored as `object`, and decimals for the
numbers.

## Setup

One library, and the helpers that build the readings.

- `pandas`, imported as `pd`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings

`year_of_readings` yields one tuple for every station at every hour of 2025, made by the same formula as
the **sqlite3, Deep Dive** guide, with Svalbard silent on 2 March. `readings` is the table of all 35,040
of them, built from that list of rows, with `time` turned into dates and times. `stations` holds each
station's latitude from `LATITUDES`, with Kirkenes, which has no readings.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


## Worked examples

### A table from a dictionary of columns

The keys name the columns, and every list gives one column's values, in row order:

In [2]:
summary = pd.DataFrame({
    "station": ["Bergen", "Oslo", "Svalbard", "Tromso"],
    "planned_mean": [8.0, 6.5, -4.5, 3.5],
    "latitude": [60.39, 59.91, 78.22, 69.65],
})
summary

,station,planned_mean,latitude
0,Bergen,8.0,60.39
1,Oslo,6.5,59.91
2,Svalbard,-4.5,78.22
3,Tromso,3.5,69.65


Nobody gave the rows labels, so the index is 0 to 3, as it was for a Series made from a list.

### Reading the shape of a table

These four tell you what a table holds without printing it:

In [3]:
print(summary.shape)
print(summary.columns.tolist())
print(summary.index)
print(summary.dtypes)

(4, 3)
['station', 'planned_mean', 'latitude']
RangeIndex(start=0, stop=4, step=1)
station          object
planned_mean    float64
latitude        float64
dtype: object


`shape` is (rows, columns). `columns` is an index of its own, holding the column names, and `tolist()`
turns it into a plain list. `dtypes` gives one type for each column. The same four on `readings`:

In [4]:
print(readings.shape)
print(readings.columns.tolist())
print(readings.index)
print(readings.dtypes)

(35040, 3)
['station', 'time', 'celsius']
RangeIndex(start=0, stop=35040, step=1)
station            object
time       datetime64[ns]
celsius           float64
dtype: object


35,040 rows and 3 columns, labeled 0 to 35039. `time` is `datetime64[ns]`, the type pandas uses for dates
and times, because Setup converted it. The **Inspecting** notebook goes further, with `info` and
`describe`.

### A table from a list of rows

A list of tuples is rows, so the column names come separately, in `columns`. Here are the first eight
tuples `year_of_readings` makes, the first two hours of the year:

In [5]:
rows = list(year_of_readings())[:8]
print(rows[:2])

first_hours = pd.DataFrame(rows, columns=["station", "time", "celsius"])
print(first_hours)
print(first_hours.dtypes)

[('Bergen', '2025-01-01T00:00', -3.6), ('Oslo', '2025-01-01T00:00', -3.5)]
    station              time  celsius
0    Bergen  2025-01-01T00:00     -3.6
1      Oslo  2025-01-01T00:00     -3.5
2  Svalbard  2025-01-01T00:00    -14.6
3    Tromso  2025-01-01T00:00     -6.7
4    Bergen  2025-01-01T01:00     -3.7
5      Oslo  2025-01-01T01:00     -5.3
6  Svalbard  2025-01-01T01:00    -16.4
7    Tromso  2025-01-01T01:00     -8.5
station     object
time        object
celsius    float64
dtype: object


This is how Setup built `readings`, from all 35,040 tuples. Here `time` is still text, `object`, because
the tuples hold it as text, and the `pd.to_datetime` line in Setup is the one that made it dates and times.

### A table from a list of records

An API that describes stations might send a dictionary for each one, and leave out a field that applies
to only one station, as `heated` applies here only to Svalbard:

In [6]:
records = [
    {"station": "Bergen", "latitude": 60.39},
    {"station": "Oslo", "latitude": 59.91},
    {"station": "Svalbard", "latitude": 78.22, "heated": True},
    {"station": "Tromso", "latitude": 69.65},
]
described = pd.DataFrame(records)

print(described)
print(described.dtypes)

    station  latitude heated
0    Bergen     60.39    NaN
1      Oslo     59.91    NaN
2  Svalbard     78.22   True
3    Tromso     69.65    NaN
station      object
latitude    float64
heated       object
dtype: object


Every key from any record became a column, in the order pandas first met it. The three records without
`heated` got `NaN` there, and the column's type is `object`, because it mixes `True` with `NaN`. The
**Missing Data** notebook covers what to do with such a column.

### A table from a NumPy array

The readings cycle through the four stations every hour, so their values reshape into an array with one
row for each hour and one column for each station. `columns` names the columns, and `index` labels the
rows with the year's 8,760 hours:

In [7]:
values = readings["celsius"].to_numpy().reshape(8760, 4)
hours = readings["time"].unique()

wide = pd.DataFrame(values, columns=list(STATIONS), index=hours)
wide.head(3)

,Bergen,Oslo,Svalbard,Tromso
2025-01-01 00:00:00,-3.6,-3.5,-14.6,-6.7
2025-01-01 01:00:00,-3.7,-5.3,-16.4,-8.5
2025-01-01 02:00:00,-3.7,-5.3,-16.4,-8.5


`unique()` returned each hour once, in the order of the table. `wide` holds the same 35,040 readings as
`readings`, arranged wide instead of long:

In [8]:
print(wide.shape)
print(wide.dtypes)
print(wide.index[:2])

(8760, 4)
Bergen      float64
Oslo        float64
Svalbard    float64
Tromso      float64
dtype: object
DatetimeIndex(['2025-01-01 00:00:00', '2025-01-01 01:00:00'], dtype='datetime64[ns]', freq=None)


An array has one type, so every column is `float64`. The index is a `DatetimeIndex`, an index of times,
which is what lets a row be found by its date. The **Reshaping** notebook gets the same table from
`readings` with `pivot`, without counting on the order of the rows.

### A column shares the table's index

A column of `wide` is a Series labeled by hour, and a column of `readings` is a Series with the table's
own index:

In [9]:
print(wide["Oslo"].head(2))
print(readings["celsius"].index.equals(readings.index))

2025-01-01 00:00:00   -3.5
2025-01-01 01:00:00   -5.3
Name: Oslo, dtype: float64
True


`equals` compares two indexes label by label. A column carries the table's labels with it, so a Series
calculated from one column still lines up with the table's rows.

### A row as a Series

`loc` with one row label returns that row, named by its label:

In [10]:
print(wide.loc["2025-07-17 15:00"])
print()
print(readings.loc[0])

Bergen      19.2
Oslo        19.3
Svalbard     8.2
Tromso      16.1
Name: 2025-07-17 15:00:00, dtype: float64

station                 Bergen
time       2025-01-01 00:00:00
celsius                   -3.6
Name: 0, dtype: object


A row of `wide` is four decimals, so its type is `float64`. A row of `readings` holds a name, a time and a
reading, and the only type that holds all three is `object`. Columns are the natural unit of a table, and
the **Performance and Memory** notebook measures what a loop over rows costs.

### Turning the table around

`.T` swaps rows and columns. On `wide`, it gives the shape the **NumPy, Deep Dive** guide used, a row for
each station and a column for each hour:

In [11]:
print(wide.T.shape)
print(wide.T.index.tolist())

(4, 8760)
['Bergen', 'Oslo', 'Svalbard', 'Tromso']


## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/03-dataframe-solutions.ipynb).

**1.** Make a table from `LATITUDES` with two columns, `station` and `latitude`, and print its shape and dtypes.

In [12]:
# your code here

**2.** Build a table from the first 12 tuples of `year_of_readings()`, and print its dtypes. Say why `time` is `object`.

In [13]:
# your code here

**3.** Make the wide table of the readings, with a column for each station and the hours as the index, and print the four readings at 03:00 on 12 January.

In [14]:
# your code here

**4.** Print how many rows and how many columns `readings` has, and its column names, without printing the table.

In [15]:
# your code here

**5.** Build a table from three records, one of which has no `latitude`, and print it. Say what pandas put in the missing place, and what type the column became.

In [16]:
# your code here

## Common errors

### ValueError: All arrays must be of the same length

In [17]:
pd.DataFrame({
    "station": ["Bergen", "Oslo", "Svalbard", "Tromso"],
    "latitude": [60.39, 59.91, 78.22],
})

ValueError: All arrays must be of the same length

A dictionary is columns, and every column of a table has one value for each row. Four names and three
latitudes cannot make a table, so the constructor raises. Count the values in each list, or build the
table from records, which allow a missing field.

### ValueError: If using all scalar values, you must pass an index

In [18]:
pd.DataFrame({"station": "Bergen", "latitude": 60.39})

ValueError: If using all scalar values, you must pass an index

With single values and no lists, pandas has no way to know how many rows to make. Wrap each value in a
list, `{"station": ["Bergen"], "latitude": [60.39]}`, or pass `index=[0]`, which says there is one row.

### ValueError: 2 columns passed, passed data had 3 columns

In [19]:
pd.DataFrame(rows, columns=["station", "celsius"])

ValueError: 2 columns passed, passed data had 3 columns

Each tuple in `rows` has three values, the station, the time and the reading, and `columns` named only
two. The message counts both sides. Give one name for every value in a row.

### No error, and a column of numbers stored as text

In [20]:
typed = pd.DataFrame([("Bergen", 60.39), ("Oslo", "59.91"), ("Tromso", 69.65)],
                     columns=["station", "latitude"])

print(typed)
print(typed.dtypes)
print(typed["latitude"].tolist())

  station latitude
0  Bergen    60.39
1    Oslo    59.91
2  Tromso    69.65
station     object
latitude    object
dtype: object
[60.39, '59.91', 69.65]


Oslo's latitude was typed with quotes, so it is text, and one text value made the whole column `object`.
The printed table shows nothing wrong. `dtypes` reports `object` for `latitude`, and the list, with quotes
around `'59.91'`, shows which value is text.
The trouble arrives with the first arithmetic:

In [21]:
typed["latitude"] + 1

TypeError: can only concatenate str (not "int") to str

pandas added 1 to each value in turn, and the text one raised. `pd.to_numeric` converts the column, and
raises on any value that is not a number, which is what you want here:

In [22]:
typed["latitude"] = pd.to_numeric(typed["latitude"])

print(typed.dtypes)

station      object
latitude    float64
dtype: object


## Recap

- Build a table from a dictionary of columns, a list of rows with `columns`, a list of records, or a NumPy
  array with `columns` and `index`
- Read `shape`, `columns`, `index` and `dtypes` before asking a table anything
- Expect a key that only some records have to become a column with `NaN` in the rest
- Spot `object` where numbers were expected, and convert the column with `pd.to_numeric`
- Take a column as a Series that shares the table's index, and a row as a Series whose type has to hold
  every column's values

## What is next

The **Reading Data** notebook builds tables from files with `read_csv`, and the arguments that decide
whether the columns and their types come out right.

---

&#8592; **Previous:** [Series](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/02-series.ipynb)  &nbsp;·&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Reading Data](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/04-reading-data.ipynb) &#8594;
